In [1]:
import numpy as np
import pandas as pd
import chromadb

In [3]:
df = pd.read_csv("../data/cleaned/cleaned_reviews.csv")
embeddings = np.load("../data/cleaned/sentence_embeddings.npy")

In [5]:
print(df.shape)
print(embeddings.shape)

(30157, 11)
(30157, 384)


In [6]:
client = chromadb.PersistentClient(path="../data/chromadb")
collection = client.get_or_create_collection(
    name="complaint_embeddings",
    metadata={"hnsw:space":"cosine"} #uses cosine similarity for search
    ) 

In [7]:
print("Collection created: ",collection.name)

Collection created:  complaint_embeddings


In [12]:
batch_size = 500

for i in range(0,len(df),batch_size):
    batch_df = df.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]

    collection.add(
        ids= [str(i) for i in batch_df.index],
        embeddings=batch_embeddings.tolist(),
        documents=batch_df["text_clean_full"].tolist(),
        metadatas= batch_df[['rating','topic_id','topic_label']].to_dict(orient='records')
    )

print("Total documents added: ",collection.count())

Total documents added:  30157


In [14]:
import re
import emoji
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
def text_preprocessing(text,remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = re.sub(r'&[a-z]+;|&#\d+;', '', text) 
    text = re.sub(r'<[^>]+>', '', text)
    text = emoji.replace_emoji(text,replace="")
    special_character_pattern = r'[^a-zA-Z0-9\s]+'
    text = re.sub(special_character_pattern,"",text)
    text = re.sub(r"\s+"," ",text).strip()
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        words = text.split(" ")
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
    
    return text

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
def retrieve(query,collection,model,top_k=3):
    query = text_preprocessing(query,remove_stopwords=False)
    query_embedding = model.encode(query).tolist()
    result = collection.query(
        query_embeddings = [query_embedding],
        n_results=top_k,
        include= ['documents','metadatas','distances']
    )
    return result

In [18]:
results = retrieve("battery not charging",collection,model,top_k=3)
print(results)

{'ids': [['29641', '6404', '10297']], 'embeddings': None, 'documents': [['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'topic_label': 'Charging issues', 'topic_id': 0, 'rating': 1.0}, {'topic_label': 'Charging issues', 'rating': 1.0, 'topic_id': 0}, {'topic_id': 0, 'topic_label': 'Charging issues', 'rating': 1.0}]], 'distances': [[0.2036428451538086, 0.2220737338066101, 0.22316217422485352]]}
